# 🎤 Speech Emotion Recognition using Deep Learning

## 📌 Objectif
Ce projet vise à reconnaître automatiquement les émotions à partir de la parole en utilisant le dataset RAVDESS.

## 🧠 Méthodologie
- Data Quality Analysis
- Feature Extraction (MFCC, ZCR, RMS)
- Data Preparation
- Modèle Deep Learning (CNN + BiLSTM)
- Évaluation (Accuracy, Loss, Confusion Matrix)

## 📊 Dataset
RAVDESS (Ryerson Audio-Visual Database of Emotional Speech and Song)

In [ ]:
import os
import glob
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Conv1D, MaxPooling1D, BatchNormalization
from tensorflow.keras.layers import LSTM, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

In [ ]:
DATA_PATH = "RAVDESS"

## 🔍 Data Quality Analysis

In [ ]:
emotion_map = {
    "01": "neutral", "02": "calm", "03": "happy", "04": "sad",
    "05": "angry", "06": "fearful", "07": "disgust", "08": "surprise"
}

data = []

for actor in os.listdir(DATA_PATH):
    actor_path = os.path.join(DATA_PATH, actor)
    if os.path.isdir(actor_path):
        for file in os.listdir(actor_path):
            if file.endswith(".wav"):
                parts = file.split("-")
                if parts[0] == "03":  # speech only
                    emotion = emotion_map.get(parts[2])
                    if emotion:
                        data.append([os.path.join(actor_path, file), emotion])

df = pd.DataFrame(data, columns=["path", "emotion"])

print("Total fichiers :", len(df))
print(df["emotion"].value_counts())

In [ ]:
plt.figure(figsize=(10,5))
sns.countplot(data=df, x="emotion", order=df["emotion"].value_counts().index)
plt.title("Distribution des émotions")
plt.xticks(rotation=45)
plt.show()

## 🎧 Visualisation du signal

In [ ]:
sample = df.iloc[0]["path"]
signal, sr = librosa.load(sample, sr=22050)

plt.figure(figsize=(12,4))
librosa.display.waveshow(signal, sr=sr)
plt.title("Waveform")
plt.show()

In [ ]:
mel = librosa.feature.melspectrogram(y=signal, sr=sr)
mel_db = librosa.power_to_db(mel)

plt.figure(figsize=(12,4))
librosa.display.specshow(mel_db, sr=sr, x_axis="time", y_axis="mel")
plt.colorbar()
plt.title("Mel Spectrogram")
plt.show()

## 🎯 Feature Extraction
MFCC + ZCR + RMS (comme recommandé dans la littérature)

In [ ]:
def extract_features(file):
    signal, sr = librosa.load(file, sr=22050, duration=3)
    
    if len(signal) < 22050*3:
        signal = np.pad(signal, (0, 22050*3 - len(signal)))
    
    mfcc = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=40)
    zcr = librosa.feature.zero_crossing_rate(y=signal)
    rms = librosa.feature.rms(y=signal)   
    
    features = np.vstack([mfcc, zcr, rms])
    
    if features.shape[1] < 130:
        features = np.pad(features, ((0,0),(0,130-features.shape[1])))
    else:
        features = features[:, :130]
    
    return features

DATA PREPARATION

In [ ]:
X, y = [], []

for _, row in df.iterrows():
    feat = extract_features(row["path"])
    X.append(feat)
    y.append(row["emotion"])

X = np.array(X)
y = np.array(y)

In [ ]:
#encoding
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)
y_cat = to_categorical(y_encoded)

In [ ]:
#reshape
X = np.transpose(X, (0, 2, 1))

In [ ]:
#Normalisation 
X = (X - np.mean(X)) / np.std(X)

In [ ]:
#split 
X_train, X_test, y_train, y_test = train_test_split(
    X, y_cat, test_size=0.2, random_state=42
)

## 🤖 Modèle Deep Learning (CNN + BiLSTM)

In [ ]:
model = Sequential([
    Conv1D(64, 3, activation='relu', input_shape=(X.shape[1], X.shape[2])),
    BatchNormalization(),
    MaxPooling1D(2),
    Dropout(0.3),

    Conv1D(128, 3, activation='relu'),
    BatchNormalization(),
    MaxPooling1D(2),
    Dropout(0.3),
# auge,tation de l'architecturecjg
    Bidirectional(LSTM(64)),
    Dropout(0.3),

    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(y_cat.shape[1], activation='softmax')
])

In [ ]:
#compilation
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
#training 
early_stop = EarlyStopping(patience=5, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=32,
    callbacks=[early_stop]
)

In [ ]:
# evaluation 
loss, acc = model.evaluate(X_test, y_test)
print("Accuracy :", acc)
print("Loss :", loss)

In [ ]:
plt.plot(history.history["accuracy"], label="train")
plt.plot(history.history["val_accuracy"], label="val")
plt.legend()
plt.title("Accuracy")
plt.show()


In [ ]:
plt.figure(figsize=(10,5))

plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")

plt.title("Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.legend()
plt.grid()

plt.show()

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# 1) Prédictions du modèle sur X_test
y_pred_prob = model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1)   # classe prédite
y_true = np.argmax(y_test, axis=1)        # vraie classe

# 2) Matrice de confusion brute
cm = confusion_matrix(y_true, y_pred)

# 3) Matrice normalisée en pourcentage par ligne
cm_percent = cm.astype("float") / cm.sum(axis=1, keepdims=True)
cm_percent = np.nan_to_num(cm_percent)  # évite les erreurs si une ligne vaut 0

# Style général plus propre
sns.set_theme(style="white", font_scale=1.0)

# =========================
# MATRICE 1 : VALEURS BRUTES
# =========================
plt.figure(figsize=(11, 8))
ax = sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="YlGnBu",          # bleu/vert doux
    linewidths=0.5,
    linecolor="white",
    cbar=True,
    xticklabels=encoder.classes_,
    yticklabels=encoder.classes_
)

plt.title("Confusion Matrix - Valeurs brutes", fontsize=15, pad=14)
plt.xlabel("Émotions prédites", fontsize=12)
plt.ylabel("Vraies émotions", fontsize=12)
plt.xticks(rotation=35, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# =========================
# MATRICE 2 : POURCENTAGES
# =========================
plt.figure(figsize=(11, 8))
ax = sns.heatmap(
    cm_percent * 100,
    annot=True,
    fmt=".1f",
    cmap="PuBuGn",          # plus doux et ergonomique
    linewidths=0.5,
    linecolor="white",
    cbar=True,
    xticklabels=encoder.classes_,
    yticklabels=encoder.classes_
)

plt.title("Confusion Matrix - Pourcentages (%)", fontsize=15, pad=14)
plt.xlabel("Émotions prédites", fontsize=12)
plt.ylabel("Vraies émotions", fontsize=12)
plt.xticks(rotation=35, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()